In [1]:
import pandas as pd
import scipy as sp 
from scipy.sparse import coo_matrix
import numpy as np

# reading data
Data downloaded from Codex [here](https://codex.flywire.ai/api/download?dataset=banc), based on [Bates et al. 2025](https://www.biorxiv.org/content/10.1101/2025.07.31.667571v2.full). 

In [2]:
conn_full = pd.read_csv("C:/Users/44745/Downloads/connections_princeton.csv.gz", compression="gzip")
meta = pd.read_csv("C:/Users/44745/Downloads/neurons.csv.gz", compression="gzip")

In [3]:
conn = conn_full.groupby(['pre_root_id','post_root_id']).syn_count.sum().reset_index()
conn

,pre_root_id,post_root_id,syn_count
0,720575941350274352,720575941399414563,30
1,720575941350274352,720575941409918831,3
2,720575941350274352,720575941425044820,3
3,720575941350274352,720575941434753568,9
4,720575941350274352,720575941438392364,3
...,...,...,...
2676587,720575941733281579,720575941617250431,3
2676588,720575941733281579,720575941623072970,5
2676589,720575941733281579,720575941640436255,6
2676590,720575941733281579,720575941645063329,5


In [4]:
allids = set(conn.pre_root_id).union(set(conn.post_root_id))
len(allids)

112885

In [5]:
meta.columns = meta.columns.str.lower().str.replace(" ", "_")
meta.columns

Index(['root_id', 'top_in/out_region', 'community_labels', 'predicted_nt_type',
       'predicted_nt_confidence', 'verified_nt_type', 'verified_neuropeptide',
       'body_part', 'function', 'flow', 'super_class', 'class', 'sub_class',
       'hemilineage', 'nerve', 'soma_side', 'primary_cell_type',
       'alternative_cell_type(s)', 'cable_length_(nm)', 'surface_area_(nm^2)',
       'volume_(nm^3)'],
      dtype='object')

In [6]:
# what's different between primary_cell_type and alternative_cell_type(s)
meta.loc[(meta["primary_cell_type"] != meta["alternative_cell_type(s)"]) & (~meta["alternative_cell_type(s)"].isna()) & (~meta["alternative_cell_type(s)"].isna()), ['primary_cell_type', 'alternative_cell_type(s)']]

,primary_cell_type,alternative_cell_type(s)
88,IN08A011,"IN08A011,IN08A018_T2_L"
100,trochanter_flexor,"trochanter_flexor,trochanter_flexor_MesoALN_po..."
337,SAD015,"SAD015,SAD018"
386,CB0810,"CB0810,antennal_motor_neuron"
396,DNfl042,"DNfl042,DNge036"
...,...,...
115034,PLP185,"PLP185,PLP186"
115059,SMP003,"SMP003,SMP005"
115078,LAL074,"LAL074,LAL084"
115080,hg2,"hg2,iv2"


In [7]:
meta.super_class.value_counts(dropna=False)

super_class
central_brain_intrinsic           26882
NaN                               26792
optic_lobe_intrinsic              25744
ventral_nerve_cord_intrinsic      12759
sensory                           12467
visual_projection                  5356
ascending                          1838
descending                         1313
motor                               831
sensory_ascending                   509
visual_centrifugal                  439
visceral_circulatory                189
glia                                 14
sensory_descending                   13
ascending_visceral_circulatory        5
Name: count, dtype: int64

# NT

In [8]:
meta.predicted_nt_type = meta.predicted_nt_type.str.lower()
meta.predicted_nt_type.value_counts(dropna=False)

predicted_nt_type
ach     65367
gaba    19856
glut    15910
NaN      5666
da       5187
ser      2352
hist      297
oct       295
tyr       221
Name: count, dtype: int64

In [9]:
meta.predicted_nt_type.replace({'ach': 'acetylcholine', 'glut': 'glutamate', 'da': 'dopamine', 'ser': 'serotonin', 'hist': 'histamine', 'oct': 'octopamine', 'tyr': 'tyramine'}, inplace=True)

C:\Users\44745\AppData\Local\Temp\ipykernel_21368\1124769434.py:1: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  meta.predicted_nt_type.replace({'ach': 'acetylcholine', 'glut': 'glutamate', 'da': 'dopamine', 'ser': 'serotonin', 'hist': 'histamine', 'oct': 'octopamine', 'tyr': 'tyramine'}, inplace=True)


In [10]:
meta.verified_nt_type.value_counts(dropna=False)

verified_nt_type
NaN                                    74583
acetylcholine                          26382
gaba                                    6438
glutamate                               4829
gaba,nitric_oxide                       1767
acetylcholine,dopamine,nitric_oxide      467
dopamine                                 205
acetylcholine,nitric_oxide               137
serotonin                                 74
octopamine                                59
histamine                                 41
dopamine,nitric_oxide                     41
tyramine                                  39
acetylcholine,histamine                   36
acetylcholine,serotonin                   19
acetylcholine,glutamate                   16
dopamine,tyramine                          4
acetylcholine,octopamine                   3
acetylcholine,gaba                         2
acetylcholine,dopamine                     2
gaba,serotonin                             2
glutamate,tyramine                    

In [11]:
# known_nt takes priority
meta.loc[:, ["known_nt_simplified"]] = meta.verified_nt_type
# pattern matching: only keep if value contains 'glutamate', 'gaba', or 'acetylcholine'
meta.loc[:, ["known_nt_simplified"]] = meta.known_nt_simplified.str.extract(
    "(glutamate|gaba|acetylcholine)", expand=False
)
# note that if a value has multiple matches, only the first one is kept
# e.g. 'glutamate, gaba' will be simplified to 'glutamate'
meta.known_nt_simplified.value_counts(dropna=False)

# then use this column to replace values in top_nt column
meta.loc[:, ["top_nt"]] = meta.known_nt_simplified.fillna(meta.predicted_nt_type)

In [12]:
# first check consistency of NT within each type
meta.loc[:, ["top_nt"]] = meta["top_nt"].fillna("unknown")
nt_count_per_type = meta.groupby("primary_cell_type")["top_nt"].nunique()
nt_count_per_type[nt_count_per_type > 1]

primary_cell_type
5-HTPMPD01    2
ADNM1         2
ADNM2         2
AN00A006      2
AN05B096      2
             ..
vDeltaM       2
vLN24         2
vLN28         2
vpoEN         2
w-cHIN        2
Name: top_nt, Length: 1630, dtype: int64

In [13]:
# For those with different NT for each neuron, get the majority NT for each type
nt_conflict_types = nt_count_per_type[nt_count_per_type > 1].index

# Getting counts of top_nt for each primary_cell_type
nt_conflict_type_counts = (
    meta[meta.primary_cell_type.isin(nt_conflict_types)]
    .groupby(["primary_cell_type", "top_nt"])
    .size()
)

# Convert the series to a DataFrame and reset index
nt_conflict_type_counts = nt_conflict_type_counts.reset_index(name="counts")

# Sort by primary_cell_type and counts in descending order
nt_conflict_type_counts.sort_values(
    by=["primary_cell_type", "counts"], ascending=[True, False], inplace=True
)

# Initialize dictionary with existing type-nt matching
type_nt = dict(zip(meta.primary_cell_type, meta.top_nt))
# Initialize a list to keep track of types with equal top_nt counts
types_with_equal_top_nt_counts = []


# Custom function to handle ties, random selection, and record keeping
def select_random_nt_and_record_ties(df):
    max_count = df["counts"].max()
    top_nts = df[df["counts"] == max_count]
    if len(top_nts) > 1:  # If there are ties
        types_with_equal_top_nt_counts.append(
            df["primary_cell_type"].iloc[0]
        )  # Record the type with ties
        random_nt = np.random.choice(top_nts["top_nt"].values)
        while random_nt == "unknown":
            random_nt = np.random.choice(top_nts["top_nt"].values)
        return random_nt  # Random selection among ties
    else:
        return top_nts["top_nt"].values[0]


# Loop through each type to get the top_nt, handling ties appropriately
for atype in nt_conflict_types:
    type_df = nt_conflict_type_counts[nt_conflict_type_counts["primary_cell_type"] == atype]
    top_nt = select_random_nt_and_record_ties(type_df)
    type_nt[atype] = top_nt

len(types_with_equal_top_nt_counts)

388

In [14]:
types_with_equal_top_nt_counts

['5-HTPMPD01',
 'ADNM1',
 'ADNM2',
 'AN05B096',
 'AN06A030',
 'AN06B023',
 'AN07B004',
 'AN09A005',
 'AN27X003',
 'AN27X009',
 'ANXXX019',
 'ANXXX434',
 'AN_GNG_SAD_6',
 'ATL026',
 'ATL037',
 'ATL043',
 'AVLP031',
 'AVLP032',
 'AVLP055',
 'AVLP084',
 'AVLP213',
 'AVLP216',
 'AVLP219',
 'AVLP281',
 'AVLP371',
 'AVLP432',
 'AVLP458',
 'AVLP506',
 'AVLP533',
 'AVLP570',
 'CB0023',
 'CB0029',
 'CB0060',
 'CB0067',
 'CB0138',
 'CB0191',
 'CB0207',
 'CB0288',
 'CB0298',
 'CB0317',
 'CB0458',
 'CB0483',
 'CB0548',
 'CB0579',
 'CB0602',
 'CB0643',
 'CB0648',
 'CB0701',
 'CB0716',
 'CB0720',
 'CB0746',
 'CB0750',
 'CB0789',
 'CB0799',
 'CB0802',
 'CB0803',
 'CB0804',
 'CB0809',
 'CB0810',
 'CB0827',
 'CB0858',
 'CB0866',
 'CB0886',
 'CB0896',
 'CB0898',
 'CB0914',
 'CB0921',
 'CB0923',
 'CB0942',
 'CB0972',
 'CB1018',
 'CB1040',
 'CB1073',
 'CB1104',
 'CB1105',
 'CB1152',
 'CB1163',
 'CB1181',
 'CB1200',
 'CB1223',
 'CB1263',
 'CB1303',
 'CB1326',
 'CB1363',
 'CB1389',
 'CB1440',
 'CB1494',
 'C

In [15]:
# the types without any known nt
[atype for atype, nt in type_nt.items() if nt == "unknown"]

['BM_InOm',
 'T1',
 'ENXXX286',
 'BM_vOcci_vPoOr',
 'KCab-m',
 'CB0991',
 'ENXXX226',
 'MNad13',
 'm_NSC_DILP',
 'MNad02',
 'm_NSC_DH44',
 'CB0873',
 'CB0845',
 'PAM08',
 'MNad43',
 'MNad54',
 'MNxm03',
 'MNad01',
 'CV1030',
 'MNad16',
 'long_tendon_muscle_B',
 'CEM',
 'OCC02b',
 'PAM01',
 'KCab-c',
 'femur_reductor_tiny',
 'CB2748',
 'tarsus_depressor_B',
 'PAM14',
 'CvN4',
 'PAM02',
 'SAD069',
 'OCC02a',
 'm_NSC_unknown',
 'l_NSC_unknown',
 'OCI',
 'MNad05',
 'MNad25',
 'MNad33',
 'CB0836',
 'l_NSC_ITP',
 'DNg08_b',
 'EN00B026',
 'CB0723',
 'PAM04_a',
 'MNad17',
 'SNpp54',
 'MNad18',
 'accessory_trochanter_flexor_MetaLN',
 'MNad24',
 'l_NSC_DH31',
 'm_NSC_DMS',
 'MNad29',
 'CB0875',
 'long_tendon_muscle_A_dipalpha',
 'PAM12',
 'CB2563',
 'EN00B015',
 'MNad03',
 'accessory_trochanter_flexor',
 'PAM13',
 'MNad14',
 'SEZ_NSC_Hugin',
 'haltere_motor_neuron_unknown',
 'long_tendon_muscle_A',
 'CB2394',
 'tarsus_depressor_A',
 'EN00B023',
 'CB0911',
 'MNad06',
 'DVM3a-b',
 'MNad09',
 'CB18

In [16]:
# update the top_nt column to be consistent within each type
meta.loc[:, ["top_nt"]] = meta.primary_cell_type.map(type_nt)
meta

,root_id,top_in/out_region,community_labels,predicted_nt_type,predicted_nt_confidence,verified_nt_type,verified_neuropeptide,body_part,function,flow,...,hemilineage,nerve,soma_side,primary_cell_type,alternative_cell_type(s),cable_length_(nm),surface_area_(nm^2),volume_(nm^3),known_nt_simplified,top_nt
0,720575940381905254,NO_CONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gaba
1,720575940386709331,NO_CONS,soma in brain,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gaba
2,720575940387110289,NO_CONS,sensory neuron,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gaba
3,720575940389711170,NO_CONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gaba
4,720575940407089948,NO_CONS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,gaba
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
115146,720575941733218347,SAD.AVLP,NaN,acetylcholine,0.87,NaN,NaN,NaN,NaN,intrinsic,...,ALl1_ventral,NaN,right,CB1817a,CB1817a,NaN,NaN,NaN,NaN,acetylcholine
115147,720575941733241899,SAD.AMMC,NaN,gaba,0.71,NaN,NaN,antenna,"auditory_high_frequency,proprioception",afferent,...,NaN,left_antennal_nerve,left,JO-B,JO-B,NaN,NaN,NaN,NaN,gaba
115148,720575941733258795,AMNP,NaN,acetylcholine,0.72,NaN,NaN,wing_margin,tactile,afferent,...,NaN,right_anterior_dorsal_mesothoracic_nerve,right,SNta11,SNta11,NaN,NaN,NaN,NaN,acetylcholine
115149,720575941733260075,ME,soma in brain,gaba,0.34,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,right,NaN,NaN,NaN,NaN,NaN,NaN,gaba


# make coo

In [17]:
# instead of making a dense matrix based on the edgelist above, let's make a sparse one from the edgelist directly
# first make a coo matrix
nodes = set(conn.pre_root_id).union(set(conn.post_root_id))
sorted_nodes = sorted(nodes)  # Convert the set to a sorted list
nodes_to_idx = {node: num for num, node in enumerate(sorted_nodes)}

# type to type connectivity
conn["pre_idx"] = conn.pre_root_id.map(nodes_to_idx)
conn["post_idx"] = conn.post_root_id.map(nodes_to_idx)

# Create COO matrix
row = conn["pre_idx"].values
col = conn["post_idx"].values
data = conn["syn_count"].values
matrix_size = len(nodes)
coo = coo_matrix((data, (row, col)), shape=(matrix_size, matrix_size))

# then turn it into csc matrix
csc = coo.tocsc()

# calculate the size
csc_size = csc.data.nbytes  # Size of the data array
csc_size += csc.indices.nbytes  # Size of the indices array
csc_size += csc.indptr.nbytes  # Size of the index pointer array
# number of MB
csc_size / 1e6, csc.shape

(32.570648, (112885, 112885))

In [18]:
csc

<Compressed Sparse Column sparse matrix of dtype 'int64'
	with 2676592 stored elements and shape (112885, 112885)>

In [19]:
# calculate the total post-synapses for each neuron
total_post = (
    conn_full[conn_full.post_root_id.isin(nodes)]
    .groupby("post_root_id")
    .syn_count.sum()
)

# some neurons have no postsynapses (receptors). Let's add those to total_post with value of 0
no_post = nodes - set(total_post.index)
no_post_dict = dict(zip(no_post, np.zeros(len(no_post), dtype=int)))
total_post = pd.concat([total_post, pd.Series(no_post_dict)])
total_post

720575941071734516      4
720575941076956631      9
720575941350274352    543
720575941350334256    231
720575941350352176    617
                     ... 
720575941543600100      0
720575941502885867      0
720575941687607279      0
720575941457592304      0
720575941668388851      0
Length: 112885, dtype: int64

In [20]:
# re-order so that it matches order of nodes
total_post = total_post.loc[sorted_nodes]
total_post

720575941071734516       4
720575941076956631       9
720575941350274352     543
720575941350334256     231
720575941350352176     617
                      ... 
720575941733218347    1492
720575941733241899       6
720575941733258795      13
720575941733260075       4
720575941733281579     104
Length: 112885, dtype: int64

In [21]:
# Handling division by zero in case some columns have a sum of zero
# that is, where a neuron doesn't have incoming synapses
col_sums_with_inversion = np.reciprocal(
    total_post.to_numpy().astype(float), where=total_post.to_numpy() != 0
)
# Multiply each column by the inverse of its sum
inprop = csc.multiply(col_sums_with_inversion)
# and then reduce the precision to float32 to save memory
inprop = inprop.astype(np.float32)

In [22]:
# calculate the total pre-synapses for each neuron
total_pre = (
    conn_full[conn_full.pre_root_id.isin(nodes)]
    .groupby("pre_root_id")
    .syn_count.sum()
)

# some neurons have no presynapses (receptors). Let's add those to total_pre with value of 0
no_pre = nodes - set(total_pre.index)
no_pre_dict = dict(zip(no_pre, np.zeros(len(no_pre), dtype=int)))
total_pre = pd.concat([total_pre, pd.Series(no_pre_dict)])
# re-order so that it matches order of nodes
total_pre = total_pre.loc[sorted_nodes]

# Handling division by zero in case some rows have a sum of zero
# that is, where a neuron doesn't have incoming synapses
row_sums_with_inversion = np.reciprocal(
    total_pre.to_numpy().astype(float), where=total_pre.to_numpy() != 0
)
# Multiply each row by the inverse of its sum
outprop = csc.multiply(row_sums_with_inversion[:, None])
# and then reduce the precision to float32 to save memory
outprop = outprop.astype(np.float32)

In [23]:
sp.sparse.save_npz(
    "../data/BANC/banc_syncount_all_neuron.npz",
    csc,)
sp.sparse.save_npz(
    "../data/BANC/banc_inprop_all_neuron.npz",
    inprop,
)
sp.sparse.save_npz(
    "../data/BANC/banc_outprop_all_neuron.npz",
    outprop,
)

# save meta

In [24]:
meta = meta[meta.root_id.isin(nodes)]
meta.loc[:, ["idx"]] = meta.root_id.map(nodes_to_idx)
# if GABA/Glu, -1, otherwise 1
meta.loc[:, ["sign"]] = meta.top_nt.map(lambda x: -1 if x in ["gaba", 'glutamate'] else 1)
meta.rename(columns={"primary_cell_type": "cell_type"}, inplace=True)
meta.to_csv("../data/BANC/banc_meta_all_neuron.csv", index=False)

C:\Users\44745\AppData\Local\Temp\ipykernel_21368\1808574263.py:5: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  meta.rename(columns={"primary_cell_type": "cell_type"}, inplace=True)
